# Chapter 8 Lab — INT8 vs FP16: Memory & Throughput Estimate

**Estimated time:** 30 minutes · **Read-and-run.** This is an *illustrative* back-of-envelope
model of what a TensorRT-LLM INT8 build buys you — not a real GPU benchmark.

In [ ]:
params = 13_000_000_000  # 13B
def weight_gb(bytes_per_param): return params*bytes_per_param/1e9
fp16_gb = weight_gb(2); int8_gb = weight_gb(1)
print(f'FP16 weights: {fp16_gb:.1f} GB')
print(f'INT8 weights: {int8_gb:.1f} GB  -> fits on a smaller / fewer GPUs')

In [ ]:
# Decoding is largely memory-bandwidth bound; a crude proxy: tokens/s ~ bandwidth / bytes_read_per_token.
gpu_bw_gbps = 2000.0  # e.g. ~2 TB/s class accelerator
def tok_per_s(weight_gb): return gpu_bw_gbps/weight_gb
base = tok_per_s(fp16_gb); opt = tok_per_s(int8_gb)
print(f'FP16 ~ {base:6.1f} tok/s (proxy)')
print(f'INT8 ~ {opt:6.1f} tok/s (proxy)  -> ~{opt/base:.1f}x')
print('\nReality check: quantization can cost accuracy; always validate quality, not just speed.')

## Key takeaway
Inference optimization is about matching **quantization precision** and **batching strategy** to
the GPU architecture and the latency SLA. There is no universally best choice — INT8/FP8 trade a
little accuracy for memory and throughput; measure both.